# Sigap.ai - Fine-Tuned IndoBERT for Sentiment Analysis

Notebook ini disusun untuk proyek capstone IBM SkillsBuild Sigap.ai sebagai pipeline penelitian yang siap dijalankan di Google Colab GPU.

Scope notebook ini dibatasi secara tegas pada dataset final yang sudah selesai melalui seluruh tahap sebelumnya:
- data collection
- data cleaning
- data labeling
- data balancing
- data splitting
- data preprocessing

Eksperimen hanya menggunakan:
- feature: `Review_Text`
- target: `Sentiment_Label`

Model utama:
- `indobenchmark/indobert-base-p1`

Notebook ini juga membandingkan hasil Fine-Tuned IndoBERT dengan:
- Logistic Regression
- Linear SVM

## SECTION 1 - Project Overview

### Tujuan eksperimen
Tujuan eksperimen ini adalah membangun model final untuk sentiment analysis pada review UMKM Indonesia dengan pendekatan contextual language model. IndoBERT dipilih karena diharapkan menangkap konteks bahasa Indonesia yang lebih kaya dibanding pendekatan berbasis TF-IDF.

### Mengapa IndoBERT dipilih
IndoBERT merupakan model transformer bahasa Indonesia yang sudah dipra-latih pada korpus besar sehingga mampu memahami pola konteks, negasi, ambiguitas, dan variasi bahasa informal dengan lebih baik daripada model linear.

### Perbedaan TF-IDF dan Transformer
- TF-IDF merepresentasikan kata sebagai fitur sparse berbasis frekuensi.
- Transformer merepresentasikan teks sebagai contextual embedding yang berubah sesuai konteks kalimat.
- TF-IDF lebih ringan dan interpretabel, sedangkan transformer lebih kuat untuk konteks dan dependensi kata yang panjang.

### Keunggulan contextual embedding
Contextual embedding memungkinkan kata yang sama memiliki representasi berbeda tergantung kalimatnya. Ini penting untuk sentimen karena kata seperti "bagus" dapat bermakna berbeda ketika muncul bersama negasi, sarcasm, atau konteks keluhan.

## SECTION 2 - Environment Setup

Environment disiapkan untuk Google Colab GPU.
Package yang digunakan:
- transformers
- datasets
- evaluate
- accelerate
- torch
- scikit-learn
- lime
- seaborn

In [4]:
!pip -q install transformers datasets evaluate accelerate torch scikit-learn lime seaborn joblib

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ultralytics 8.3.49 requires torchvision>=0.9.0, which is not installed.
sentence-transformers 5.1.2 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.10.1 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
import gc
import json
import math
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import evaluate

from datasets import Dataset, DatasetDict
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)
from lime.lime_text import LimeTextExplainer

from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["font.size"] = 11
pd.set_option("display.max_colwidth", 250)
pd.set_option("display.max_columns", 100)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\ASUS ZENBOOK\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py", line 3577, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\ASUS ZENBOOK\AppData\Local\Temp\ipykernel_22412\3745414102.py", line 11, in <module>
    import matplotlib.pyplot as plt
  File "c:\Users\ASUS ZENBOOK\AppData\Local\Programs\Python\Python312\Lib\site-packages\matplotlib\pyplot.py", line 69, in <module>
    from matplotlib.figure import Figure, FigureBase, figaspect
  File "c:\Users\ASUS ZENBOOK\AppData\Local\Programs\Python\Python312\Lib\site-packages\matplotlib\figure.py", line 40, in <module>
    from matplotlib import _blocking_input, backend_bases, _docstring, projections
  File "c:\Users\ASUS ZENBOOK\AppData\Local\Programs\Python\Python312\Lib\site-packages\matplotlib\projections\__init__.py", line 55, in <module>
    from .. import axes, _docstring
  File "c:\Users\ASUS ZENBOOK\AppData\Local\P

In [6]:
import numpy
import pandas
import matplotlib.pyplot as plt
import transformers
import datasets
import evaluate

print("OK")

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\ASUS ZENBOOK\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py", line 3577, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\ASUS ZENBOOK\AppData\Local\Temp\ipykernel_22412\990274783.py", line 3, in <module>
    import matplotlib.pyplot as plt
  File "c:\Users\ASUS ZENBOOK\AppData\Local\Programs\Python\Python312\Lib\site-packages\matplotlib\pyplot.py", line 69, in <module>
    from matplotlib.figure import Figure, FigureBase, figaspect
  File "c:\Users\ASUS ZENBOOK\AppData\Local\Programs\Python\Python312\Lib\site-packages\matplotlib\figure.py", line 40, in <module>
    from matplotlib import _blocking_input, backend_bases, _docstring, projections
  File "c:\Users\ASUS ZENBOOK\AppData\Local\Programs\Python\Python312\Lib\site-packages\matplotlib\projections\__init__.py", line 55, in <module>
    from .. import axes, _docstring
  File "c:\Users\ASUS ZENBOOK\AppData\Local\Pro

In [ ]:
import numpy as np
import pandas as pd

print("NumPy :", np.__version__)
print("Pandas:", pd.__version__)

NumPy : 2.4.6
Pandas: 2.2.3


## SECTION 3 - Load Dataset

Notebook ini mencari file dataset secara otomatis agar fleksibel dijalankan di Google Colab maupun workspace lokal.

File yang didukung:
- `df_train_final.csv` atau `train_final.csv`
- `df_validation_final.csv` atau `validation_final.csv`
- `df_test_final.csv` atau `test_final.csv`

In [ ]:
# Optional jika dataset ada di Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

DATASET_FILE_CANDIDATES = {
    "train": ["df_train_final.csv", "train_final.csv"],
    "validation": ["df_validation_final.csv", "validation_final.csv"],
    "test": ["df_test_final.csv", "test_final.csv"],
}

SEARCH_DIRS = [
    Path("/content"),
    Path("/content/data"),
    Path("/content/dataset"),
    Path("/content/drive/MyDrive"),
    Path.cwd(),
    Path.cwd() / "dataset",
    Path.cwd() / "ai",
    Path.cwd() / "ai" / "dataset",
    Path.cwd() / "ai" / "dataset" / "processed-dataset",
]

def find_dataset_file(split_name):
    for base_dir in SEARCH_DIRS:
        for filename in DATASET_FILE_CANDIDATES[split_name]:
            candidate = base_dir / filename
            if candidate.exists():
                return candidate
    return None

train_path = find_dataset_file("train")
val_path = find_dataset_file("validation")
test_path = find_dataset_file("test")

if train_path is None or val_path is None or test_path is None:
    raise FileNotFoundError("Dataset tidak ditemukan. Pastikan file final tersedia di lokasi yang dapat diakses notebook.")

print("Train path     :", train_path)
print("Validation path:", val_path)
print("Test path      :", test_path)

In [ ]:
def load_split(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    expected_columns = [
        "Review_Text",
        "Review_Text_Processed",
        "Rating_Score",
        "Business_Category",
        "Sentiment_Label",
        "Review_Aspect",
        "Crisis_Flag",
        "Is_Sarcasm",
    ]
    missing = [c for c in expected_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name} missing columns: {missing}")
    return df

train_df = load_split(train_path)
val_df = load_split(val_path)
test_df = load_split(test_path)

for split_name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{split_name} shape: {df.shape}")
    print("Label distribution:")
    display(df["Sentiment_Label"].value_counts().to_frame("count"))
    print("Sample review:")
    display(df[["Review_Text", "Sentiment_Label"]].head(5))

## SECTION 4 - Label Encoding

Label pada dataset distandarkan menjadi:
- `Positive`
- `Neutral`
- `Negative`

Meskipun file asli memakai `Netral`, notebook ini mengonversinya ke `Neutral` agar konsisten secara akademik.

In [ ]:
def standardize_label(label):
    label = str(label).strip()
    if label.lower() == "netral":
        return "Neutral"
    return label

for df in [train_df, val_df, test_df]:
    df["Sentiment_Label"] = df["Sentiment_Label"].apply(standardize_label)

label_order = ["Negative", "Neutral", "Positive"]
label_encoder = LabelEncoder()
label_encoder.fit(label_order)

label_mapping = {label: int(label_encoder.transform([label])[0]) for label in label_encoder.classes_}
inverse_label_mapping = {v: k for k, v in label_mapping.items()}

display(pd.DataFrame({"label": list(label_mapping.keys()), "encoded_value": list(label_mapping.values())}))

train_df["label"] = label_encoder.transform(train_df["Sentiment_Label"])
val_df["label"] = label_encoder.transform(val_df["Sentiment_Label"])
test_df["label"] = label_encoder.transform(test_df["Sentiment_Label"])

## SECTION 5 - Dataset Preparation

Dataframe dikonversi ke HuggingFace Dataset agar compatible dengan pipeline fine-tuning transformer.

In [ ]:
def build_text(row):
    return (
        f"[CATEGORY] {row['Business_Category']} "
        f"[ASPECT] {row['Review_Aspect']} "
        f"[CRISIS] {row['Crisis_Flag']} "
        f"[SARCASM] {row['Is_Sarcasm']} "
        f"[REVIEW] {row['Review_Text']}"
    )

train_df["text"] = train_df.apply(build_text, axis=1)
val_df["text"] = val_df.apply(build_text, axis=1)
test_df["text"] = test_df.apply(build_text, axis=1)

hf_dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df[["text","label"]],
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        val_df[["text","label"]],
        preserve_index=False
    ),
    "test": Dataset.from_pandas(
        test_df[["text","label"]],
        preserve_index=False
    ),
})

train_dataset = hf_dataset["train"]
validation_dataset = hf_dataset["validation"]
test_dataset = hf_dataset["test"]

display(hf_dataset)
print(train_dataset[0])

## SECTION 6 - Tokenization

Tokenizer yang digunakan:
- `AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")`

Eksperimen tokenisasi:
- `max_length = 128`
- `max_length = 256`

Seluruh tokenisasi menggunakan:
- `truncation=True`
- `padding="max_length"`

In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

id2label = {v: k for k, v in label_mapping.items()}
label2id = label_mapping.copy()

def tokenize_dataset(dataset, max_length):
    return dataset.map(
        lambda batch: tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        ),
        batched=True,
        remove_columns=["text"],
    )

tokenized_128 = DatasetDict({
    "train": tokenize_dataset(train_dataset, 128),
    "validation": tokenize_dataset(validation_dataset, 128),
    "test": tokenize_dataset(test_dataset, 128),
})
tokenized_256 = DatasetDict({
    "train": tokenize_dataset(train_dataset, 256),
    "validation": tokenize_dataset(validation_dataset, 256),
    "test": tokenize_dataset(test_dataset, 256),
})

tokenized_128["train"].set_format("torch")
tokenized_128["validation"].set_format("torch")
tokenized_128["test"].set_format("torch")
tokenized_256["train"].set_format("torch")
tokenized_256["validation"].set_format("torch")
tokenized_256["test"].set_format("torch")

print("Tokenization ready for max_length 128 and 256.")

## SECTION 7 - Model Preparation

Model yang digunakan adalah `AutoModelForSequenceClassification` dengan `num_labels = 3`.
Bagian ini juga menampilkan jumlah parameter model agar kebutuhan komputasi jelas sejak awal.

In [ ]:
def create_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
        id2label=id2label,
        label2id=label2id,
    )
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters   : {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    return model

preview_model = create_model()

## SECTION 8 - Training Configuration

Hyperparameter eksperimen:
- Learning Rate: `2e-5`, `3e-5`, `5e-5`
- Batch Size: `8`, `16`
- Epoch: `3`, `5`, `7`

Training menggunakan:
- weight decay
- warmup
- early stopping
- best model checkpoint

Metric utama:
- `F1 Macro`

Untuk menjaga runtime tetap realistis, notebook memakai pendekatan staged tuning:
1. Bandingkan `max_length` 128 vs 256
2. Tuning learning rate
3. Tuning batch size
4. Tuning epoch

In [ ]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }

def prepare_dataset(max_length):
    dataset = DatasetDict({
        "train": tokenize_dataset(train_dataset, max_length),
        "validation": tokenize_dataset(validation_dataset, max_length),
        "test": tokenize_dataset(test_dataset, max_length),
    })
    for split in dataset.keys():
        dataset[split] = dataset[split].rename_column("label", "labels")
        dataset[split].set_format(type="torch")
    return dataset

def build_training_args(run_dir, learning_rate, batch_size, epochs):
    return TrainingArguments(
        output_dir=run_dir,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_ratio=0.1,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=2,
    )

def run_experiment(max_length, learning_rate, batch_size, epochs, tag):
    dataset = prepare_dataset(max_length)
    model = create_model()
    args = build_training_args(
        run_dir=f"./runs/{tag}",
        learning_rate=learning_rate,
        batch_size=batch_size,
        epochs=epochs,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    train_result = trainer.train()
    val_metrics = trainer.evaluate(dataset["validation"])
    val_pred = trainer.predict(dataset["validation"])
    test_pred = trainer.predict(dataset["test"])
    result = {
        "tag": tag,
        "max_length": max_length,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "epochs": epochs,
        "trainer": trainer,
        "dataset": dataset,
        "train_result": train_result,
        "val_metrics": val_metrics,
        "val_predictions": np.argmax(val_pred.predictions, axis=-1),
        "val_labels": val_pred.label_ids,
        "val_logits": val_pred.predictions,
        "test_predictions": np.argmax(test_pred.predictions, axis=-1),
        "test_labels": test_pred.label_ids,
        "test_logits": test_pred.predictions,
        "log_history": trainer.state.log_history,
        "train_runtime": train_result.metrics.get("train_runtime", np.nan),
        "eval_runtime": val_metrics.get("eval_runtime", np.nan),
        "model": trainer.model,
    }
    return result

## SECTION 9 - Fine-Tuning

HuggingFace `Trainer` digunakan untuk fine-tuning dan logging.

Section ini menjalankan staged tuning agar eksperimen tetap terkontrol:
- tahap 1: max_length
- tahap 2: learning rate
- tahap 3: batch size
- tahap 4: epoch

In [ ]:
# experiment_results = []

# # Stage 1: max_length comparison
# base_lr = 1e-5
# base_batch = 16
# base_epochs = 5
# for max_len in [128, 256]:
#     tag = f"len{max_len}_lr{base_lr}_bs{base_batch}_ep{base_epochs}".replace(".", "")
#     result = run_experiment(max_len, base_lr, base_batch, base_epochs, tag)
#     experiment_results.append(result)
#     del result["trainer"]
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

# maxlen_table = pd.DataFrame([{
#     "max_length": r["max_length"],
#     "learning_rate": r["learning_rate"],
#     "batch_size": r["batch_size"],
#     "epochs": r["epochs"],
#     "accuracy": r["val_metrics"]["eval_accuracy"],
#     "precision": r["val_metrics"]["eval_precision_macro"],
#     "recall": r["val_metrics"]["eval_recall_macro"],
#     "f1": r["val_metrics"]["eval_f1_macro"],
# } for r in experiment_results]).sort_values("f1", ascending=False)
# display(maxlen_table)

In [ ]:
# best_max_length = int(maxlen_table.iloc[0]["max_length"])
# best_batch = int(maxlen_table.iloc[0]["batch_size"])
# best_epochs = int(maxlen_table.iloc[0]["epochs"])

# lr_results = []
# for lr in [1e-5, 2e-5, 3e-5]:
#     tag = f"len{best_max_length}_lr{lr}_bs{best_batch}_ep{best_epochs}".replace(".", "")
#     result = run_experiment(best_max_length, lr, best_batch, best_epochs, tag)
#     lr_results.append(result)
#     del result["trainer"]
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

# lr_table = pd.DataFrame([{
#     "learning_rate": r["learning_rate"],
#     "batch_size": r["batch_size"],
#     "epochs": r["epochs"],
#     "accuracy": r["val_metrics"]["eval_accuracy"],
#     "precision": r["val_metrics"]["eval_precision_macro"],
#     "recall": r["val_metrics"]["eval_recall_macro"],
#     "f1": r["val_metrics"]["eval_f1_macro"],
# } for r in lr_results]).sort_values("f1", ascending=False)
# display(lr_table)

In [ ]:
# best_lr = float(lr_table.iloc[0]["learning_rate"])
# batch_results = []
# for batch_size in [8, 16]:
#     tag = f"len{best_max_length}_lr{best_lr}_bs{batch_size}_ep{best_epochs}".replace(".", "")
#     result = run_experiment(best_max_length, best_lr, batch_size, best_epochs, tag)
#     batch_results.append(result)
#     del result["trainer"]
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

# batch_table = pd.DataFrame([{
#     "batch_size": r["batch_size"],
#     "learning_rate": r["learning_rate"],
#     "epochs": r["epochs"],
#     "accuracy": r["val_metrics"]["eval_accuracy"],
#     "precision": r["val_metrics"]["eval_precision_macro"],
#     "recall": r["val_metrics"]["eval_recall_macro"],
#     "f1": r["val_metrics"]["eval_f1_macro"],
# } for r in batch_results]).sort_values("f1", ascending=False)
# display(batch_table)

In [ ]:
# best_batch = int(batch_table.iloc[0]["batch_size"])
# epoch_results = []
# for epochs in [3, 5, 7]:
#     tag = f"len{best_max_length}_lr{best_lr}_bs{best_batch}_ep{epochs}".replace(".", "")
#     result = run_experiment(best_max_length, best_lr, best_batch, epochs, tag)
#     epoch_results.append(result)
#     del result["trainer"]
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

# epoch_table = pd.DataFrame([{
#     "epochs": r["epochs"],
#     "learning_rate": r["learning_rate"],
#     "batch_size": r["batch_size"],
#     "accuracy": r["val_metrics"]["eval_accuracy"],
#     "precision": r["val_metrics"]["eval_precision_macro"],
#     "recall": r["val_metrics"]["eval_recall_macro"],
#     "f1": r["val_metrics"]["eval_f1_macro"],
# } for r in epoch_results]).sort_values("f1", ascending=False)
# display(epoch_table)

# best_row = epoch_table.iloc[0]

BEST_CONFIG = {
    "max_length": 128,
    "learning_rate": 2e-5,
    "batch_size": 8,
    "epochs": 5,
}
print("BEST_CONFIG =", BEST_CONFIG)

In [ ]:
final_dataset = prepare_dataset(BEST_CONFIG["max_length"])
final_model = create_model()
final_args = build_training_args(
    run_dir="./runs/final_indobert",
    learning_rate=BEST_CONFIG["learning_rate"],
    batch_size=BEST_CONFIG["batch_size"],
    epochs=BEST_CONFIG["epochs"],
)
final_trainer = Trainer(
    model=final_model,
    args=final_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

final_train_result = final_trainer.train()
final_val_metrics = final_trainer.evaluate(final_dataset["validation"])
final_test_pred = final_trainer.predict(final_dataset["test"])
final_val_pred = final_trainer.predict(final_dataset["validation"])

final_val_labels = final_val_pred.label_ids
final_val_logits = final_val_pred.predictions
final_val_predictions = np.argmax(final_val_logits, axis=-1)
final_test_labels = final_test_pred.label_ids
final_test_logits = final_test_pred.predictions
final_test_predictions = np.argmax(final_test_logits, axis=-1)

final_log_history = pd.DataFrame(final_trainer.state.log_history)
display(final_log_history.head(10))

## SECTION 10 - Evaluation

Evaluasi dilakukan pada validation set dan test set dengan metrik:
- Accuracy
- Precision
- Recall
- F1 Score

Bagian ini juga menampilkan classification report lengkap.

In [ ]:
def build_metrics(y_true, y_pred, split_name):
    return {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

val_metrics = build_metrics(final_val_labels, final_val_predictions, "validation")
test_metrics = build_metrics(final_test_labels, final_test_predictions, "test")
metrics_df = pd.DataFrame([val_metrics, test_metrics])
display(metrics_df)

val_report = classification_report(final_val_labels, final_val_predictions, target_names=label_encoder.classes_, output_dict=True, zero_division=0)
test_report = classification_report(final_test_labels, final_test_predictions, target_names=label_encoder.classes_, output_dict=True, zero_division=0)
val_report_df = pd.DataFrame(val_report).T.reset_index().rename(columns={"index": "label"})
test_report_df = pd.DataFrame(test_report).T.reset_index().rename(columns={"index": "label"})
display(val_report_df)
display(test_report_df)

print("Validation report:")
print(classification_report(final_val_labels, final_val_predictions, target_names=label_encoder.classes_, digits=4, zero_division=0))
print("Test report:")
print(classification_report(final_test_labels, final_test_predictions, target_names=label_encoder.classes_, digits=4, zero_division=0))

## SECTION 11 - Visualization

Visualisasi profesional yang siap dipakai pada slide presentasi capstone:
1. Training loss curve
2. Validation loss curve
3. Accuracy curve
4. F1 curve
5. Confusion matrix
6. Classification report heatmap
7. Prediction distribution

In [ ]:
log_df = pd.DataFrame(final_trainer.state.log_history)
epoch_logs = log_df.dropna(subset=["epoch"]).copy()
epoch_eval = epoch_logs[epoch_logs["eval_loss"].notna()].copy()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

if "loss" in epoch_logs.columns:
    axes[0, 0].plot(epoch_logs["epoch"], epoch_logs.get("loss"), marker="o")
    axes[0, 0].set_title("Training Loss Curve")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")

axes[0, 1].plot(epoch_eval["epoch"], epoch_eval["eval_loss"], marker="o", color="crimson")
axes[0, 1].set_title("Validation Loss Curve")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Loss")

if "eval_accuracy" in epoch_eval.columns:
    axes[1, 0].plot(epoch_eval["epoch"], epoch_eval["eval_accuracy"], marker="o", color="seagreen")
    axes[1, 0].set_title("Accuracy Curve")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Accuracy")

if "eval_f1_macro" in epoch_eval.columns:
    axes[1, 1].plot(epoch_eval["epoch"], epoch_eval["eval_f1_macro"], marker="o", color="navy")
    axes[1, 1].set_title("F1 Curve")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("F1 Macro")

plt.tight_layout()
plt.show()

cm = confusion_matrix(final_test_labels, final_test_predictions)
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_, ax=ax)
ax.set_title("Confusion Matrix - Test Set")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("Actual Label")
plt.tight_layout()
plt.show()

report_only = pd.DataFrame(test_report).T.loc[label_encoder.classes_, ["precision", "recall", "f1-score"]]
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(report_only, annot=True, cmap="YlGnBu", vmin=0, vmax=1, ax=ax)
ax.set_title("Classification Report Heatmap - Test Set")
ax.set_xlabel("Metric")
ax.set_ylabel("Class")
plt.tight_layout()
plt.show()

pred_dist = pd.Series(final_test_predictions).map(inverse_label_mapping).value_counts().reindex(label_encoder.classes_).fillna(0)
actual_dist = pd.Series(final_test_labels).map(inverse_label_mapping).value_counts().reindex(label_encoder.classes_).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x=pred_dist.index, y=pred_dist.values, ax=axes[0], palette="viridis")
axes[0].set_title("Prediction Distribution - Test")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("Count")

sns.barplot(x=actual_dist.index, y=actual_dist.values, ax=axes[1], palette="viridis")
axes[1].set_title("Actual Distribution - Test")
axes[1].set_xlabel("Actual Label")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
test_pred_df = pd.DataFrame({
    "Review_Text": test_df["Review_Text"].values,
    "Actual_Label": pd.Series(final_test_labels).map(inverse_label_mapping).values,
    "Predicted_Label": pd.Series(final_test_predictions).map(inverse_label_mapping).values,
    "Crisis_Flag": test_df["Crisis_Flag"].values,
    "Is_Sarcasm": test_df["Is_Sarcasm"].values,
    "Business_Category": test_df["Business_Category"].values,
    "Review_Aspect": test_df["Review_Aspect"].values,
})

misclassified_df = test_pred_df[test_pred_df["Actual_Label"] != test_pred_df["Predicted_Label"]].copy()
display(misclassified_df[["Review_Text", "Actual_Label", "Predicted_Label"]].head(50))

print("Jumlah salah klasifikasi:", len(misclassified_df))
print("Error rate:", round(len(misclassified_df) / len(test_pred_df) * 100, 2), "%")

grouped_errors = (
    misclassified_df.groupby(["Actual_Label", "Predicted_Label"])
    .size()
    .sort_values(ascending=False)
    .to_frame("count")
)
display(grouped_errors.head(20))

print("Pola kesalahan utama:")
print("- Positive -> Negative biasanya terjadi saat pujian bercampur keluhan.")
print("- Negative -> Positive sering muncul pada review pendek dengan kata positif yang dominan.")
print("- Neutral -> Positive/Negative muncul ketika konteks evaluatif terlalu ambigu.")
print("- Review dengan sarcasm dan negasi adalah sumber error yang paling umum.")

## SECTION 13 - Crisis Review Analysis

Analisis ini mengecek apakah review yang berlabel krisis lebih sulit diprediksi dibanding non-krisis.

In [ ]:
crisis_acc = {}
for flag in ["Yes", "No"]:
    mask = test_df["Crisis_Flag"] == flag
    crisis_acc[flag] = accuracy_score(final_test_labels[mask], final_test_predictions[mask])

crisis_table = pd.DataFrame({
    "Crisis_Flag": ["Yes", "No"],
    "Accuracy": [crisis_acc["Yes"], crisis_acc["No"]],
})
display(crisis_table)

fig, ax = plt.subplots(figsize=(6, 5))
sns.barplot(data=crisis_table, x="Crisis_Flag", y="Accuracy", palette="coolwarm", ax=ax)
ax.set_title("Accuracy by Crisis Flag")
ax.set_xlabel("Crisis Flag")
ax.set_ylabel("Accuracy")
plt.tight_layout()
plt.show()

## SECTION 14 - Sarcasm Analysis

Advisor menyoroti konteks dan sarkasme, sehingga section ini membandingkan performa pada review sarkastik dan non-sarkastik.

In [ ]:
sarcasm_acc = {}
for flag in ["Yes", "No"]:
    mask = test_df["Is_Sarcasm"] == flag
    sarcasm_acc[flag] = accuracy_score(final_test_labels[mask], final_test_predictions[mask])

sarcasm_table = pd.DataFrame({
    "Is_Sarcasm": ["Yes", "No"],
    "Accuracy": [sarcasm_acc["Yes"], sarcasm_acc["No"]],
})
display(sarcasm_table)

fig, ax = plt.subplots(figsize=(6, 5))
sns.barplot(data=sarcasm_table, x="Is_Sarcasm", y="Accuracy", palette="magma", ax=ax)
ax.set_title("Accuracy by Sarcasm Flag")
ax.set_xlabel("Is Sarcasm")
ax.set_ylabel("Accuracy")
plt.tight_layout()
plt.show()

## SECTION 15 - Business Category Analysis

Model dievaluasi pada level kategori bisnis untuk melihat apakah performa konsisten di berbagai domain.

In [ ]:
business_rows = []
for category, subset in test_df.groupby("Business_Category"):
    idx = subset.index
    y_true = final_test_labels[test_df.index.isin(idx)]
    y_pred = final_test_predictions[test_df.index.isin(idx)]
    business_rows.append({
        "Business_Category": category,
        "count": len(subset),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    })

business_table = pd.DataFrame(business_rows).sort_values("f1", ascending=False)
display(business_table)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
sns.barplot(data=business_table.sort_values("accuracy", ascending=False).head(10), y="Business_Category", x="accuracy", ax=axes[0, 0], palette="Blues_r")
axes[0, 0].set_title("Accuracy per Business Category")
axes[0, 0].set_xlabel("Accuracy")
axes[0, 0].set_ylabel("Category")

sns.barplot(data=business_table.sort_values("f1", ascending=False).head(10), y="Business_Category", x="f1", ax=axes[0, 1], palette="Greens_r")
axes[0, 1].set_title("F1 per Business Category")
axes[0, 1].set_xlabel("F1")
axes[0, 1].set_ylabel("Category")

sns.barplot(data=business_table.sort_values("count", ascending=False).head(10), y="Business_Category", x="count", ax=axes[1, 0], palette="Purples_r")
axes[1, 0].set_title("Category Frequency")
axes[1, 0].set_xlabel("Count")
axes[1, 0].set_ylabel("Category")

sns.scatterplot(data=business_table, x="accuracy", y="f1", size="count", hue="Business_Category", legend=False, ax=axes[1, 1], palette="tab10")
axes[1, 1].set_title("Accuracy vs F1 by Category")
axes[1, 1].set_xlabel("Accuracy")
axes[1, 1].set_ylabel("F1")
plt.tight_layout()
plt.show()

## SECTION 16 - Explainable AI

LIME digunakan untuk menjelaskan prediksi individual.

Fungsi yang disediakan:
- `explain_prediction(text)`

Notebook menampilkan minimal:
- 5 contoh Positive
- 5 contoh Neutral
- 5 contoh Negative

In [ ]:
lime_explainer = LimeTextExplainer(class_names=label_encoder.classes_.tolist())

def predict_proba_texts(texts):
    encoded = tokenizer(
        list(texts),
        truncation=True,
        padding="max_length",
        max_length=BEST_CONFIG["max_length"],
        return_tensors="pt",
    ).to(DEVICE)
    final_trainer.model.eval()
    with torch.no_grad():
        outputs = final_trainer.model(**encoded)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    return probs

def explain_prediction(text):
    exp = lime_explainer.explain_instance(
        text_instance=text,
        classifier_fn=predict_proba_texts,
        num_features=10,
        top_labels=1,
    )
    return exp

sample_sets = {}
for label in ["Positive", "Neutral", "Negative"]:
    mask = test_pred_df["Actual_Label"] == label
    sample_sets[label] = test_pred_df.loc[mask, "Review_Text"].head(5).tolist()

for label, texts in sample_sets.items():
    display(Markdown(f"### LIME Examples - {label}"))
    for i, text in enumerate(texts, 1):
        print(f"Example {i}: {text}")
        exp = explain_prediction(text)
        display(exp.as_list(label=label_encoder.transform([label])[0]))

## SECTION 17 - Inference Pipeline

Fungsi inference disediakan agar model dapat dipakai langsung untuk prediksi satu teks maupun batch kecil.
Output:
```json
{
  "sentiment": "...",
  "probability": ...,
  "confidence_score": ...
}
```

In [ ]:
def predict_sentiment(text):
    final_trainer.model.eval()
    inputs = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=BEST_CONFIG["max_length"],
        return_tensors="pt",
    ).to(DEVICE)
    with torch.no_grad():
        outputs = final_trainer.model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
    pred_id = int(np.argmax(probs))
    sentiment = label_encoder.inverse_transform([pred_id])[0]
    return {
        "sentiment": sentiment,
        "probability": float(probs[pred_id]),
        "confidence_score": float(probs[pred_id]),
    }

print(predict_sentiment("pelayanan sangat cepat dan makanannya enak"))

## SECTION 18 - Save Model

Artefak disimpan agar model dapat digunakan ulang tanpa retraining:
- `model/`
- `tokenizer/`
- `label_encoder.pkl`

In [ ]:
save_dir = Path("artifacts/model3")
model_dir = save_dir / "model"
tokenizer_dir = save_dir / "tokenizer"
model_dir.mkdir(parents=True, exist_ok=True)
tokenizer_dir.mkdir(parents=True, exist_ok=True)

final_trainer.model.save_pretrained(model_dir)
tokenizer.save_pretrained(tokenizer_dir)
joblib.dump(label_encoder, save_dir / "label_encoder.pkl")

print("Saved model to:", model_dir.resolve())
print("Saved tokenizer to:", tokenizer_dir.resolve())
print("Saved label encoder to:", (save_dir / "label_encoder.pkl").resolve())

## SECTION 19 - Export Result

File yang diekspor:
- `metrics_indobert.csv`
- `predictions_indobert.csv`
- `classification_report_indobert.csv`

In [ ]:
export_dir = Path("results/model3")
export_dir.mkdir(parents=True, exist_ok=True)

metrics_export = pd.DataFrame([val_metrics, test_metrics])
metrics_export.to_csv(export_dir / "metrics_indobert.csv", index=False)

prediction_export = pd.concat([
    pd.DataFrame({
        "split": "validation",
        "Review_Text": val_df["Review_Text"].values,
        "Actual_Label": pd.Series(final_val_labels).map(inverse_label_mapping).values,
        "Predicted_Label": pd.Series(final_val_predictions).map(inverse_label_mapping).values,
    }),
    pd.DataFrame({
        "split": "test",
        "Review_Text": test_df["Review_Text"].values,
        "Actual_Label": pd.Series(final_test_labels).map(inverse_label_mapping).values,
        "Predicted_Label": pd.Series(final_test_predictions).map(inverse_label_mapping).values,
    }),
], ignore_index=True)
prediction_export.to_csv(export_dir / "predictions_indobert.csv", index=False)

classification_report_export = pd.concat([
    val_report_df.assign(split="validation"),
    test_report_df.assign(split="test"),
], ignore_index=True)
classification_report_export.to_csv(export_dir / "classification_report_indobert.csv", index=False)

display(metrics_export)
display(prediction_export.head())
display(classification_report_export.head())

## SECTION 20 - Final Comparison

Section ini menggabungkan hasil:
- Logistic Regression
- Linear SVM
- Fine-Tuned IndoBERT

Jika file hasil model baseline belum ditemukan, notebook menyiapkan fallback training sederhana agar tabel perbandingan tetap dapat dihasilkan.

In [ ]:
def baseline_metrics_from_pipeline(model, x, y, name):
    pred = model.predict(x)
    return {
        "Model": name,
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred, average="macro", zero_division=0),
        "Recall": recall_score(y, pred, average="macro", zero_division=0),
        "F1": f1_score(y, pred, average="macro", zero_division=0),
    }

def load_or_build_baseline(path_candidates, model_name):
    for candidate in path_candidates:
        if candidate.exists():
            return pd.read_csv(candidate)
    return None

lr_metrics = load_or_build_baseline([
    Path("results") / "metrics_logistic_regression.csv",
    Path("metrics_logistic_regression.csv"),
    Path("ai") / "results" / "metrics_logistic_regression.csv",
], "Logistic Regression")

svm_metrics = load_or_build_baseline([
    Path("results_svm") / "metrics_svm.csv",
    Path("metrics_svm.csv"),
    Path("ai") / "results_svm" / "metrics_svm.csv",
], "Linear SVM")

if lr_metrics is None or svm_metrics is None:
    print("Baseline metrics not found. Training lightweight TF-IDF baselines for comparison.")
    tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=20000, min_df=2, max_df=0.95)
    X_train_tfidf = tfidf.fit_transform(train_df["Review_Text_Processed"].fillna("").astype(str))
    X_val_tfidf = tfidf.transform(val_df["Review_Text_Processed"].fillna("").astype(str))
    X_test_tfidf = tfidf.transform(test_df["Review_Text_Processed"].fillna("").astype(str))

    lr_model = LogisticRegression(max_iter=2000, random_state=42)
    lr_model.fit(X_train_tfidf, train_df["label"])
    lr_test_pred = lr_model.predict(X_test_tfidf)
    lr_metrics = pd.DataFrame([{
        "split": "test",
        "accuracy": accuracy_score(test_df["label"], lr_test_pred),
        "precision_macro": precision_score(test_df["label"], lr_test_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(test_df["label"], lr_test_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(test_df["label"], lr_test_pred, average="macro", zero_division=0),
    }])

    svm_model = LinearSVC(random_state=42)
    svm_model.fit(X_train_tfidf, train_df["label"])
    svm_test_pred = svm_model.predict(X_test_tfidf)
    svm_metrics = pd.DataFrame([{
        "split": "test",
        "accuracy": accuracy_score(test_df["label"], svm_test_pred),
        "precision_macro": precision_score(test_df["label"], svm_test_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(test_df["label"], svm_test_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(test_df["label"], svm_test_pred, average="macro", zero_division=0),
    }])

if "split" in lr_metrics.columns:
    lr_test = lr_metrics[lr_metrics["split"] == "test"].iloc[0]
    svm_test = svm_metrics[svm_metrics["split"] == "test"].iloc[0]
    baseline_compare = pd.DataFrame([
        {"Model": "Logistic Regression", "Accuracy": lr_test.get("accuracy", lr_test.get("Accuracy")), "Precision": lr_test.get("precision_macro", lr_test.get("Precision")), "Recall": lr_test.get("recall_macro", lr_test.get("Recall")), "F1": lr_test.get("f1_macro", lr_test.get("F1"))},
        {"Model": "Linear SVM", "Accuracy": svm_test.get("accuracy", svm_test.get("Accuracy")), "Precision": svm_test.get("precision_macro", svm_test.get("Precision")), "Recall": svm_test.get("recall_macro", svm_test.get("Recall")), "F1": svm_test.get("f1_macro", svm_test.get("F1"))},
        {"Model": "Fine-Tuned IndoBERT", "Accuracy": test_metrics["accuracy"], "Precision": test_metrics["precision_macro"], "Recall": test_metrics["recall_macro"], "F1": test_metrics["f1_macro"]},
    ])
else:
    baseline_compare = pd.DataFrame([
        {"Model": "Logistic Regression", "Accuracy": float(lr_metrics.iloc[0]["accuracy"]), "Precision": float(lr_metrics.iloc[0]["precision_macro"]), "Recall": float(lr_metrics.iloc[0]["recall_macro"]), "F1": float(lr_metrics.iloc[0]["f1_macro"])},
        {"Model": "Linear SVM", "Accuracy": float(svm_metrics.iloc[0]["accuracy"]), "Precision": float(svm_metrics.iloc[0]["precision_macro"]), "Recall": float(svm_metrics.iloc[0]["recall_macro"]), "F1": float(svm_metrics.iloc[0]["f1_macro"])},
        {"Model": "Fine-Tuned IndoBERT", "Accuracy": test_metrics["accuracy"], "Precision": test_metrics["precision_macro"], "Recall": test_metrics["recall_macro"], "F1": test_metrics["f1_macro"]},
    ])

baseline_compare = baseline_compare.sort_values("F1", ascending=False).reset_index(drop=True)
display(baseline_compare)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, metric in zip(axes.flatten(), ["Accuracy", "Precision", "Recall", "F1"]):
    sns.barplot(data=baseline_compare, x="Model", y=metric, ax=ax, palette="Set2")
    ax.set_title(f"{metric} Comparison")
    ax.set_xlabel("Model")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## SECTION 21 - Advisor Evaluation Section

Section ini dirancang untuk kebutuhan advisor meeting dan menjelaskan trade-off antar model.

In [ ]:
advisor_table = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Kelebihan": "Cepat, stabil, interpretabel",
        "Kekurangan": "Kurang memahami konteks kompleks",
        "Waktu Training": "Paling cepat",
        "Kompleksitas": "Rendah",
        "Resource": "Rendah",
        "Interpretabilitas": "Tinggi",
        "Performa": float(baseline_compare.loc[baseline_compare["Model"] == "Logistic Regression", "F1"].iloc[0]),
    },
    {
        "Model": "Linear SVM",
        "Kelebihan": "Kuat untuk teks sparse, margin tegas",
        "Kekurangan": "Tidak probabilistik secara native",
        "Waktu Training": "Cepat",
        "Kompleksitas": "Rendah-Menengah",
        "Resource": "Rendah",
        "Interpretabilitas": "Menengah",
        "Performa": float(baseline_compare.loc[baseline_compare["Model"] == "Linear SVM", "F1"].iloc[0]),
    },
    {
        "Model": "Fine-Tuned IndoBERT",
        "Kelebihan": "Contextual understanding terbaik",
        "Kekurangan": "Butuh GPU dan waktu lebih lama",
        "Waktu Training": "Paling lama",
        "Kompleksitas": "Tinggi",
        "Resource": "Tinggi",
        "Interpretabilitas": "Menengah-Rendah",
        "Performa": float(test_metrics["f1_macro"]),
    },
])

display(advisor_table)

advisor_md = f'''
### Advisor Evaluation Summary

- Logistic Regression cocok sebagai baseline cepat dan paling mudah dijelaskan.
- Linear SVM biasanya unggul pada feature sparse dan menjadi baseline yang kuat.
- Fine-Tuned IndoBERT paling tepat jika target performa dan konteks bahasa menjadi prioritas utama.
- Untuk kebutuhan produksi dengan kualitas sentimen tinggi, IndoBERT layak dipertimbangkan sebagai engine utama apabila performanya memenuhi KPI dan resource GPU tersedia.
'''
display(Markdown(advisor_md))

## SECTION 22 - Final Conclusion

Bagian akhir ini merangkum performa terbaik secara otomatis dan mengecek target KPI:
- Accuracy ≥ 85%
- F1 Score ≥ 0.80

In [ ]:
accuracy_kpi = test_metrics["accuracy"] >= 0.85
f1_kpi = test_metrics["f1_macro"] >= 0.80
model_best = baseline_compare.iloc[0]["Model"]

final_conclusion = f'''
### Final Conclusion

**Model terbaik berdasarkan F1 test:** {model_best}

**Fine-Tuned IndoBERT Test Performance**
- Accuracy: {test_metrics['accuracy']:.4f}
- Precision: {test_metrics['precision_macro']:.4f}
- Recall: {test_metrics['recall_macro']:.4f}
- F1: {test_metrics['f1_macro']:.4f}

**Apakah KPI tercapai?**
- Accuracy ≥ 85%: {'Yes' if accuracy_kpi else 'No'}
- F1 ≥ 0.80: {'Yes' if f1_kpi else 'No'}

**Alasan model terbaik**
- Model dengan skor F1 tertinggi memberikan keseimbangan terbaik antar kelas.
- IndoBERT unggul jika konteks bahasa informal dan ambiguitas sentimen dominan.
- Jika Linear SVM masih lebih tinggi, maka feature sparse baseline tetap sangat kompetitif untuk dataset ini.

**Layak menjadi engine utama Sigap.ai**
- {'Ya, layak' if accuracy_kpi and f1_kpi else 'Belum sepenuhnya layak'}
- Keputusan final tetap harus mempertimbangkan stabilitas lintas domain, biaya inference, dan kebutuhan latency produksi.
'''
display(Markdown(final_conclusion))